# Notebook 02 — Batch Processing
## NYC Taxi Trips — Big Data Pipeline

**Objetivo:** Limpiar, transformar y enriquecer los datos usando PySpark. Guardar resultados en Parquet (capa processed) y cargar tablas en PostgreSQL.

**Operaciones:**
- Limpieza de datos (nulos, outliers, duplicados)
- Transformaciones y feature engineering
- Join con Taxi Zones
- Agregaciones por zona, hora y tipo de pago
- Guardar en Parquet procesado
- Cargar a PostgreSQL

In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["TMP"] = r"C:\tmp"
os.environ["TEMP"] = r"C:\tmp"

os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.enabled=false pyspark-shell"

In [3]:
# ─── Celda 1: Imports y rutas ─────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

BASE_PATH      = os.path.abspath("../")
RAW_PATH       = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
SCREENS_PATH   = os.path.join(BASE_PATH, "screenshots")

os.makedirs(PROCESSED_PATH, exist_ok=True)

DB_URL  = "jdbc:postgresql://localhost:5432/nyc_taxi_db"
DB_PROPS = {
    "user":   "postgres",
    "password": "",
    "driver": "org.postgresql.Driver"
}

print("Rutas configuradas ✅")
print("PROCESSED_PATH:", PROCESSED_PATH)

Rutas configuradas ✅
PROCESSED_PATH: c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\processed


In [4]:
# ─── Celda 2: Iniciar Spark ───────────────────────────────────────────────────
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

# Descargar driver PostgreSQL JDBC
jdbc_jar = os.path.join(BASE_PATH, "postgresql-42.7.3.jar")
if not os.path.exists(jdbc_jar):
    import urllib.request
    print("Descargando driver JDBC PostgreSQL...")
    urllib.request.urlretrieve(
        "https://jdbc.postgresql.org/download/postgresql-42.7.3.jar",
        jdbc_jar
    )
    print("Driver descargado ✅")
else:
    print("Driver JDBC ya existe ✅")

spark = SparkSession.builder \
    .appName("NYC_Taxi_BatchProcessing") \
    .master("local[2]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .config("spark.jars", jdbc_jar) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Spark OK ✅")

Driver JDBC ya existe ✅
Spark version: 3.5.1
Spark OK ✅


In [5]:
# ─── Celda 3: Cargar datos raw con pandas ─────────────────────────────────────
print("Cargando archivos Parquet...")

parquet_files = [
    os.path.join(RAW_PATH, "yellow_tripdata_2023-01.parquet"),
    os.path.join(RAW_PATH, "yellow_tripdata_2023-02.parquet"),
    os.path.join(RAW_PATH, "yellow_tripdata_2023-03.parquet"),
]

pdf = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)

# Convertir fechas a string para Spark
pdf["tpep_pickup_datetime"]  = pdf["tpep_pickup_datetime"].astype(str)
pdf["tpep_dropoff_datetime"] = pdf["tpep_dropoff_datetime"].astype(str)

print(f"Filas cargadas: {len(pdf):,}")
print("Carga completa ✅")

Cargando archivos Parquet...
Filas cargadas: 9,384,487
Carga completa ✅


In [6]:
# ─── Celda 4: Limpieza de datos con pandas ────────────────────────────────────
print(f"Filas antes de limpieza : {len(pdf):,}")

# 1. Eliminar duplicados
pdf = pdf.drop_duplicates()

# 2. Eliminar filas con nulos en columnas críticas
pdf = pdf.dropna(subset=["trip_distance", "fare_amount", "total_amount",
                          "PULocationID", "DOLocationID"])

# 3. Filtrar valores inválidos (outliers)
pdf = pdf[
    (pdf["trip_distance"]   > 0)    & (pdf["trip_distance"]   < 200)  &
    (pdf["fare_amount"]     > 0)    & (pdf["fare_amount"]     < 1000) &
    (pdf["total_amount"]    > 0)    & (pdf["total_amount"]    < 1000) &
    (pdf["passenger_count"] >= 1)   & (pdf["passenger_count"] <= 8)
]

# 4. Resetear índice
pdf = pdf.reset_index(drop=True)

print(f"Filas después de limpieza: {len(pdf):,}")
print(f"Filas eliminadas         : {9384487 - len(pdf):,}")
print("Limpieza completada ✅")

Filas antes de limpieza : 9,384,487
Filas después de limpieza: 8,807,442
Filas eliminadas         : 577,045
Limpieza completada ✅


In [7]:
# ─── Celda 5: Feature Engineering ────────────────────────────────────────────
print("Generando nuevas features...")

pdf["pickup_dt"]  = pd.to_datetime(pdf["tpep_pickup_datetime"])
pdf["dropoff_dt"] = pd.to_datetime(pdf["tpep_dropoff_datetime"])

# Features de tiempo
pdf["pickup_hour"]    = pdf["pickup_dt"].dt.hour
pdf["pickup_day"]     = pdf["pickup_dt"].dt.day_name()
pdf["pickup_month"]   = pdf["pickup_dt"].dt.month
pdf["pickup_weekday"] = pdf["pickup_dt"].dt.weekday  # 0=lunes, 6=domingo
pdf["is_weekend"]     = (pdf["pickup_weekday"] >= 5).astype(int)

# Duración del viaje en minutos
pdf["trip_duration_min"] = (
    (pdf["dropoff_dt"] - pdf["pickup_dt"]).dt.total_seconds() / 60
).round(2)

# Filtrar duraciones inválidas
pdf = pdf[(pdf["trip_duration_min"] > 0) & (pdf["trip_duration_min"] < 300)]

# Velocidad promedio (mph)
pdf["avg_speed_mph"] = (
    pdf["trip_distance"] / (pdf["trip_duration_min"] / 60)
).round(2)
pdf = pdf[pdf["avg_speed_mph"] < 150]  # eliminar velocidades imposibles

# Costo por milla
pdf["cost_per_mile"] = (pdf["fare_amount"] / pdf["trip_distance"]).round(2)
pdf = pdf[pdf["cost_per_mile"] < 100]

# Categoría de hora
def hour_category(h):
    if 6 <= h < 12:   return "Morning"
    elif 12 <= h < 17: return "Afternoon"
    elif 17 <= h < 21: return "Evening"
    else:              return "Night"

pdf["time_of_day"] = pdf["pickup_hour"].apply(hour_category)

# Tipo de pago legible
payment_map = {1: "Credit Card", 2: "Cash", 3: "No Charge", 4: "Dispute", 5: "Unknown", 6: "Voided"}
pdf["payment_type_name"] = pdf["payment_type"].map(payment_map).fillna("Unknown")

print(f"Features generadas. Filas finales: {len(pdf):,}")
print("Nuevas columnas:", ["pickup_hour","pickup_day","pickup_month",
                            "is_weekend","trip_duration_min","avg_speed_mph",
                            "cost_per_mile","time_of_day","payment_type_name"])
print("Feature engineering completado ✅")

Generando nuevas features...
Features generadas. Filas finales: 8,777,907
Nuevas columnas: ['pickup_hour', 'pickup_day', 'pickup_month', 'is_weekend', 'trip_duration_min', 'avg_speed_mph', 'cost_per_mile', 'time_of_day', 'payment_type_name']
Feature engineering completado ✅


In [11]:
# ─── Celda 6: Join con Taxi Zones ─────────────────────────────────────────────
print("Cargando Taxi Zones...")

zones_path = os.path.join(RAW_PATH, "NYC_Taxi_Zones.csv")
pdf_zones  = pd.read_csv(zones_path)

print("Columnas de zones:", list(pdf_zones.columns))
print(pdf_zones.head(3))

# Detectar columna de ID automáticamente
id_col = [c for c in pdf_zones.columns if "locationid" in c.lower() or "location_id" in c.lower()]
if not id_col:
    id_col = [pdf_zones.columns[0]]
id_col = id_col[0]
print(f"\nColumna ID detectada: {id_col}")

# Renombrar para el join
pdf_zones = pdf_zones.rename(columns={"Location ID": "LocationID"})
pdf_zones["LocationID"] = pd.to_numeric(pdf_zones["LocationID"], errors="coerce")

# Join pickup zone
pdf = pdf.merge(
    pdf_zones[["LocationID","Zone","Borough"]].rename(
        columns={"Zone": "pickup_zone", "Borough": "pickup_borough"}),
    left_on="PULocationID", right_on="LocationID", how="left"
).drop(columns=["LocationID"])

# Join dropoff zone
pdf = pdf.merge(
    pdf_zones[["LocationID","Zone","Borough"]].rename(
        columns={"Zone": "dropoff_zone", "Borough": "dropoff_borough"}),
    left_on="DOLocationID", right_on="LocationID", how="left"
).drop(columns=["LocationID"])

print(f"\nJoin completado. Filas: {len(pdf):,}")
print("Columnas nuevas: pickup_zone, pickup_borough, dropoff_zone, dropoff_borough")
print(pdf[["pickup_zone","pickup_borough","dropoff_zone","dropoff_borough"]].head(3))
print("Join con Taxi Zones completado ✅")

Cargando Taxi Zones...
Columnas de zones: ['Shape Geometry', 'Shape Length', 'Shape Area', 'Zone', 'Location ID', 'Borough']
                                      Shape Geometry  Shape Length  \
0  MULTIPOLYGON (((-74.18445299999996 40.69499599...      0.116357   
1  MULTIPOLYGON (((-73.82337597260663 40.63898704...      0.433470   
2  MULTIPOLYGON (((-73.84792614099985 40.87134223...      0.084341   

   Shape Area                     Zone  Location ID Borough  
0    0.000782           Newark Airport            1     EWR  
1    0.004866              Jamaica Bay            2  Queens  
2    0.000314  Allerton/Pelham Gardens            3   Bronx  

Columna ID detectada: Shape Geometry

Join completado. Filas: 8,780,868
Columnas nuevas: pickup_zone, pickup_borough, dropoff_zone, dropoff_borough
      pickup_zone pickup_borough           dropoff_zone dropoff_borough
0  Midtown Center      Manhattan        Lenox Hill West       Manhattan
1    Central Park      Manhattan  Upper East Side Sou

In [12]:
# ─── Celda 7: Guardar capa processed en Parquet ───────────────────────────────
print("Guardando capa processed...")

# Seleccionar columnas finales limpias
cols_to_save = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount",
    "pickup_hour", "pickup_day", "pickup_month", "pickup_weekday",
    "is_weekend", "trip_duration_min", "avg_speed_mph",
    "cost_per_mile", "time_of_day", "payment_type_name",
    "pickup_zone", "pickup_borough", "dropoff_zone", "dropoff_borough"
]

pdf_clean = pdf[cols_to_save].copy()

out_path = os.path.join(PROCESSED_PATH, "trips_processed.parquet")
pdf_clean.to_parquet(out_path, index=False)

size_mb = os.path.getsize(out_path) / (1024*1024)
print(f"Guardado en  : {out_path}")
print(f"Tamaño       : {size_mb:.1f} MB")
print(f"Filas        : {len(pdf_clean):,}")
print("Parquet procesado guardado ✅")

Guardando capa processed...
Guardado en  : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\processed\trips_processed.parquet
Tamaño       : 228.2 MB
Filas        : 8,780,868
Parquet procesado guardado ✅


In [13]:
# ─── Celda 8: Agregaciones ────────────────────────────────────────────────────
print("Generando agregaciones...")

# Agrupación por hora
agg_hourly = pdf_clean.groupby("pickup_hour").agg(
    total_trips    = ("trip_distance", "count"),
    avg_fare       = ("fare_amount",   "mean"),
    avg_distance   = ("trip_distance", "mean"),
    avg_duration   = ("trip_duration_min", "mean"),
    total_revenue  = ("total_amount",  "sum")
).round(2).reset_index()

# Agrupación por borough de origen
agg_borough = pdf_clean.groupby("pickup_borough").agg(
    total_trips   = ("trip_distance",    "count"),
    avg_fare      = ("fare_amount",      "mean"),
    avg_distance  = ("trip_distance",    "mean"),
    total_revenue = ("total_amount",     "sum"),
    avg_tip       = ("tip_amount",       "mean")
).round(2).reset_index()

# Agrupación por tipo de pago
agg_payment = pdf_clean.groupby("payment_type_name").agg(
    total_trips  = ("trip_distance", "count"),
    avg_fare     = ("fare_amount",   "mean"),
    avg_tip      = ("tip_amount",    "mean")
).round(2).reset_index()

# Agrupación por día de semana
agg_day = pdf_clean.groupby("pickup_day").agg(
    total_trips   = ("trip_distance", "count"),
    avg_fare      = ("fare_amount",   "mean"),
    total_revenue = ("total_amount",  "sum")
).round(2).reset_index()

print("\n=== Trips por hora (top 5) ===")
print(agg_hourly.sort_values("total_trips", ascending=False).head())

print("\n=== Trips por borough ===")
print(agg_borough.sort_values("total_trips", ascending=False))

print("\n=== Trips por tipo de pago ===")
print(agg_payment)

print("\nAgregaciones completadas ✅")

Generando agregaciones...

=== Trips por hora (top 5) ===
    pickup_hour  total_trips  avg_fare  avg_distance  avg_duration  \
18           18       629769     17.18          2.95         14.63   
17           17       601778     18.58          3.23         16.27   
19           19       564672     17.56          3.20         13.96   
16           16       552761     19.65          3.48         17.18   
15           15       551133     19.64          3.44         17.28   

    total_revenue  
18    17053049.87  
17    17294779.23  
19    15552458.28  
16    16561150.39  
15    15416176.54  

=== Trips por borough ===
  pickup_borough  total_trips  avg_fare  avg_distance  total_revenue  avg_tip
3      Manhattan      7832114     15.06          2.41   1.800216e+08     2.95
4         Queens       801293     52.82         12.94   5.729858e+07     8.42
1       Brooklyn        39717     25.51          5.17   1.249644e+06     2.59
0          Bronx         9672     29.53          6.68   3.2257

In [14]:
# ─── Celda 9: Cargar a PostgreSQL ─────────────────────────────────────────────
import psycopg2

print("Conectando a PostgreSQL...")
conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

# Crear tablas
cur.execute("""
DROP TABLE IF EXISTS trips_processed CASCADE;
CREATE TABLE trips_processed (
    id                   SERIAL PRIMARY KEY,
    vendor_id            INT,
    pickup_datetime      TEXT,
    dropoff_datetime     TEXT,
    passenger_count      FLOAT,
    trip_distance        FLOAT,
    payment_type         INT,
    fare_amount          FLOAT,
    tip_amount           FLOAT,
    total_amount         FLOAT,
    pickup_hour          INT,
    pickup_day           TEXT,
    pickup_month         INT,
    is_weekend           INT,
    trip_duration_min    FLOAT,
    avg_speed_mph        FLOAT,
    cost_per_mile        FLOAT,
    time_of_day          TEXT,
    payment_type_name    TEXT,
    pickup_zone          TEXT,
    pickup_borough       TEXT,
    dropoff_zone         TEXT,
    dropoff_borough      TEXT
);
""")

cur.execute("""
DROP TABLE IF EXISTS agg_hourly CASCADE;
CREATE TABLE agg_hourly (
    pickup_hour    INT,
    total_trips    INT,
    avg_fare       FLOAT,
    avg_distance   FLOAT,
    avg_duration   FLOAT,
    total_revenue  FLOAT
);
""")

cur.execute("""
DROP TABLE IF EXISTS agg_borough CASCADE;
CREATE TABLE agg_borough (
    pickup_borough TEXT,
    total_trips    INT,
    avg_fare       FLOAT,
    avg_distance   FLOAT,
    total_revenue  FLOAT,
    avg_tip        FLOAT
);
""")

cur.execute("""
DROP TABLE IF EXISTS agg_payment CASCADE;
CREATE TABLE agg_payment (
    payment_type_name TEXT,
    total_trips       INT,
    avg_fare          FLOAT,
    avg_tip           FLOAT
);
""")

conn.commit()
print("Tablas creadas ✅")

# Insertar datos de agregaciones (rápido)
from psycopg2.extras import execute_values

execute_values(cur, "INSERT INTO agg_hourly VALUES %s",
    [tuple(r) for r in agg_hourly.itertuples(index=False)])

execute_values(cur, "INSERT INTO agg_borough VALUES %s",
    [tuple(r) for r in agg_borough.itertuples(index=False)])

execute_values(cur, "INSERT INTO agg_payment VALUES %s",
    [tuple(r) for r in agg_payment.itertuples(index=False)])

conn.commit()
print("Agregaciones insertadas ✅")

# Insertar muestra de trips (10,000 filas para evidencia)
print("Insertando muestra de trips...")
sample_trips = pdf_clean[[
    "VendorID","tpep_pickup_datetime","tpep_dropoff_datetime",
    "passenger_count","trip_distance","payment_type",
    "fare_amount","tip_amount","total_amount",
    "pickup_hour","pickup_day","pickup_month","is_weekend",
    "trip_duration_min","avg_speed_mph","cost_per_mile",
    "time_of_day","payment_type_name",
    "pickup_zone","pickup_borough","dropoff_zone","dropoff_borough"
]].head(10000)

execute_values(cur,
    """INSERT INTO trips_processed 
       (vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,
        payment_type,fare_amount,tip_amount,total_amount,pickup_hour,pickup_day,
        pickup_month,is_weekend,trip_duration_min,avg_speed_mph,cost_per_mile,
        time_of_day,payment_type_name,pickup_zone,pickup_borough,dropoff_zone,dropoff_borough)
       VALUES %s""",
    [tuple(r) for r in sample_trips.itertuples(index=False)],
    page_size=500
)

conn.commit()
cur.close()
conn.close()

print("10,000 trips insertados en PostgreSQL ✅")
print("\n✅ Notebook 02 completado exitosamente.")

Conectando a PostgreSQL...
Tablas creadas ✅
Agregaciones insertadas ✅
Insertando muestra de trips...
10,000 trips insertados en PostgreSQL ✅

✅ Notebook 02 completado exitosamente.


In [15]:
# ─── Celda 10: Verificar datos en PostgreSQL ──────────────────────────────────
import psycopg2

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

tablas = ["trips_processed", "agg_hourly", "agg_borough", "agg_payment"]
for tabla in tablas:
    cur.execute(f"SELECT COUNT(*) FROM {tabla}")
    count = cur.fetchone()[0]
    print(f"  {tabla:<25} → {count:,} filas")

print("\nTop 5 boroughs por viajes:")
cur.execute("SELECT pickup_borough, total_trips FROM agg_borough ORDER BY total_trips DESC LIMIT 5")
for row in cur.fetchall():
    print(f"  {row[0]:<20} {row[1]:,}")

cur.close()
conn.close()
print("\n✅ Verificación PostgreSQL completada")

spark.stop()
print("SparkSession cerrada.")

  trips_processed           → 10,000 filas
  agg_hourly                → 24 filas
  agg_borough               → 6 filas
  agg_payment               → 4 filas

Top 5 boroughs por viajes:
  Manhattan            7,832,114
  Queens               801,293
  Brooklyn             39,717
  Bronx                9,672
  Staten Island        646

✅ Verificación PostgreSQL completada
SparkSession cerrada.
